# Semana 11: Tareas Avanzadas de NLP: NER, Question Answering, Summarization
## Notebook Conceptual (NB1) – Pipelines de Hugging Face

**Propósito:** Aplicar modelos pre-entrenados a tareas específicas usando pipelines de Hugging Face y analizar los resultados.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Usar pipeline("ner") para extraer entidades nombradas.
- Usar pipeline("question-answering") con contexto y pregunta.
- Usar pipeline("summarization") en un párrafo.
- Analizar resultados y umbrales de confianza.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalamos transformers si es necesario
try:
    import transformers
    print("Transformers ya instalado.")
except ImportError:
    !pip install transformers
    !pip install torch

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
import torch

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Texto Inventado con Entidades

Definimos un texto que contiene diversas entidades nombradas para usar en NER.

In [ ]:
# Texto con entidades
texto_ner = """
Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cupertino, California. 
The company's CEO, Tim Cook, announced the new iPhone 15 on September 12, 2023, 
at the Steve Jobs Theater. Microsoft, led by Satya Nadella, is based in Redmond, 
Washington. Google's headquarters, known as the Googleplex, is located in Mountain View.
"""

print("=== TEXTO PARA NER ===")
print(texto_ner)

---
## 2. Named Entity Recognition (NER) con Pipeline

Usamos el pipeline de NER con un modelo pre-entrenado para reconocimiento de entidades.

In [ ]:
# Cargar pipeline de NER
print("Cargando pipeline de NER...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

# Aplicar NER al texto
entities = ner_pipeline(texto_ner)

print("\n=== ENTIDADES DETECTADAS ===")
for entity in entities:
    print(f"Entidad: {entity['word']:20} | Tipo: {entity['entity_group']:10} | Score: {entity['score']:.4f} | Posición: {entity['start']}-{entity['end']}")

### 2.1. Análisis de resultados NER

Observamos que el modelo detecta correctamente:
- **ORG**: Apple, Microsoft, Google
- **PER**: Steve Jobs, Steve Wozniak, Tim Cook, Satya Nadella
- **LOC**: Cupertino, California, Redmond, Washington, Mountain View
- **DATE**: September 12, 2023

También podemos analizar los scores de confianza.

In [ ]:
# Analizar distribución de scores
scores = [entity['score'] for entity in entities]

plt.figure(figsize=(10, 4))
plt.hist(scores, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Score de confianza')
plt.ylabel('Frecuencia')
plt.title('Distribución de scores en NER')
plt.axvline(np.mean(scores), color='red', linestyle='--', label=f'Media: {np.mean(scores):.3f}')
plt.legend()
plt.show()

print(f"Score promedio: {np.mean(scores):.4f}")
print(f"Score mínimo: {min(scores):.4f}")
print(f"Score máximo: {max(scores):.4f}")

### 2.2. Filtrando por umbral de confianza

Podemos filtrar entidades con score bajo para mejorar la precisión.

In [ ]:
umbral = 0.9
entities_filtradas = [e for e in entities if e['score'] >= umbral]

print(f"Entidades con score >= {umbral}:")
for entity in entities_filtradas:
    print(f"{entity['word']:20} -> {entity['entity_group']} (score: {entity['score']:.4f})")

print(f"\nSe eliminaron {len(entities) - len(entities_filtradas)} entidades con score bajo.")

---
## 3. Question Answering con Pipeline

Usamos el pipeline de question-answering para responder preguntas sobre un contexto.

In [ ]:
# Definir contexto y preguntas
context = texto_ner

questions = [
    "Who founded Apple?",
    "Who is the CEO of Apple?",
    "Where is Google headquartered?",
    "When was the iPhone 15 announced?",
    "Who is the CEO of Microsoft?"
]

# Cargar pipeline de QA
print("Cargando pipeline de Question Answering...")
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=0 if torch.cuda.is_available() else -1)

print("\n=== PREGUNTAS Y RESPUESTAS ===\n")
for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Pregunta: {question}")
    print(f"Respuesta: {result['answer']}")
    print(f"Confianza: {result['score']:.4f}")
    print(f"Posición: {result['start']}-{result['end']}\n")

### 3.1. Análisis de resultados QA

Observamos que el modelo responde correctamente a preguntas directas. La confianza es alta (>0.95) para respuestas claras.

In [ ]:
# Experimentar con preguntas más complejas
preguntas_complejas = [
    "What did Tim Cook announce?",
    "Where is the Steve Jobs Theater?"
]

print("=== PREGUNTAS COMPLEJAS ===\n")
for question in preguntas_complejas:
    result = qa_pipeline(question=question, context=context)
    print(f"Pregunta: {question}")
    print(f"Respuesta: {result['answer']}")
    print(f"Confianza: {result['score']:.4f}\n")

---
## 4. Summarization con Pipeline

Usamos el pipeline de summarization para generar resúmenes de párrafos.

In [ ]:
# Texto largo para resumir
texto_largo = """
El cambio climático es uno de los mayores desafíos que enfrenta la humanidad en el siglo XXI.
Las actividades humanas, especialmente la quema de combustibles fósiles, han aumentado
la concentración de gases de efecto invernadero en la atmósfera, provocando un aumento
de la temperatura global. Este fenómeno, conocido como calentamiento global, tiene
consecuencias graves como el derretimiento de los glaciares, el aumento del nivel del mar,
eventos climáticos extremos más frecuentes e intensos, y la pérdida de biodiversidad.
Los científicos advierten que si no se toman medidas urgentes para reducir las emisiones,
los efectos podrían ser irreversibles. Acuerdos internacionales como el Acuerdo de París
buscan limitar el aumento de la temperatura a menos de 2 grados Celsius respecto a niveles
preindustriales. Sin embargo, las contribuciones actuales de los países son insuficientes
para alcanzar este objetivo. Se necesita una transición hacia energías renovables,
mejoras en eficiencia energética, y cambios en los patrones de consumo.
"""

print("=== TEXTO ORIGINAL ===")
print(texto_largo)

# Cargar pipeline de summarization
print("\nCargando pipeline de Summarization...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if torch.cuda.is_available() else -1)

In [ ]:
# Generar resumen
summary = summarizer(texto_largo, max_length=100, min_length=30, do_sample=False)

print("\n=== RESUMEN GENERADO ===")
print(summary[0]['summary_text'])

### 4.1. Experimentar con diferentes longitudes de resumen

In [ ]:
longitudes = [(30, 80), (50, 120), (80, 200)]

print("=== RESUMENES CON DIFERENTES LONGITUDES ===\n")
for min_len, max_len in longitudes:
    summary = summarizer(texto_largo, max_length=max_len, min_length=min_len, do_sample=False)
    print(f"Longitud {min_len}-{max_len}: {len(summary[0]['summary_text'].split())} palabras")
    print(f"Resumen: {summary[0]['summary_text']}")
    print()

### 4.2. Análisis del resumen abstractivo

El resumen generado es abstractivo, lo que significa que parafrasea y condensa la información, no solo extrae oraciones. Observamos que:
- Captura la idea principal (cambio climático, causas, consecuencias).
- Menciona el Acuerdo de París y la necesidad de transición energética.
- Es fluido y coherente.

**Desafío**: A veces puede introducir información no presente (alucinaciones) o perder detalles importantes.

---
## 5. Comparación de Tareas

Resumimos las características de cada tarea.

In [ ]:
comparacion = pd.DataFrame({
    'Tarea': ['NER', 'Question Answering', 'Summarization'],
    'Tipo': ['Extracción', 'Extracción', 'Generación'],
    'Entrada': ['Texto', 'Contexto + Pregunta', 'Texto'],
    'Salida': ['Entidades etiquetadas', 'Span de respuesta', 'Texto resumido'],
    'Métrica típica': ['F1 por entidad', 'Exact Match / F1', 'ROUGE']
})

print("=== COMPARACIÓN DE TAREAS ===")
comparacion

---
## 6. Ejercicio para el Estudiante

1. **NER**: Prueba con un texto en español (por ejemplo, una noticia) usando un modelo NER en español como `"BSC-Loeschke/distilbert-base-spanish-mlm-ner"`.

2. **QA**: Inventa un contexto sobre un tema que conozcas y haz preguntas. Observa si el modelo responde correctamente.

3. **Summarization**: Prueba con diferentes textos (artículos de noticias, emails, etc.) y evalúa cualitativamente los resúmenes.

4. **Umbrales**: Experimenta con diferentes umbrales de confianza en NER y observa cómo cambia la precisión/recall.

In [ ]:
# Espacio para el estudiante
# ...

---
## 7. Conclusiones

En este notebook hemos explorado tres tareas avanzadas de NLP usando pipelines de Hugging Face:

✔️ **NER (Named Entity Recognition)**:
- Detectamos personas, organizaciones, lugares y fechas.
- Analizamos la distribución de scores de confianza.
- Filtramos por umbral para mejorar precisión.

✔️ **Question Answering (QA)**:
- Respondimos preguntas sobre un contexto dado.
- Observamos que el modelo es preciso para preguntas directas.

✔️ **Summarization**:
- Generamos resúmenes abstractivos de textos largos.
- Experimentamos con diferentes longitudes.

**Lección clave**: Los pipelines de Hugging Face facilitan enormemente la aplicación de modelos avanzados a tareas complejas, permitiendo obtener resultados de calidad con pocas líneas de código.

En el próximo notebook (NB2) realizaremos fine-tuning de BERT para una tarea específica (NER o QA).

---
**Fin del Notebook Conceptual - Semana 11 (NLP)**